# Vet Clinics – Cleaning & Normalization (v1)

This notebook transforms the joined OSM + LOR dataset into a clean,
normalized export aligned with the unified POI schema.

Key principles:
- Preserve geometry for future spatial joins
- Keep missing values as NULL
- Enforce stable identifiers and data types
- Export a reduced, non-redundant schema for downstream use

## 1. Imports & helpers

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def to_str(series):
    return series.astype("string")

def pad_four_digits(series):
    return (
        series
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(4)
    )

def clean_postcode(series):
    return (
        series
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
    )

## 2. Load v0 dataset

In [2]:
df = pd.read_csv(
    "scripts/vets_osm_berlin_with_lor_latest_v0.csv",
    dtype={"neighborhood_id": "string", "district_id": "string"},
)

## 3. Identifier normalization

In [3]:
df_clean = pd.DataFrame()

df_clean["id"] = (
    df["id"]
    .astype("Int64")
    .astype("string")
)

## 4. Clinic name

In [4]:
name = to_str(df["name"])
operator = to_str(df["operator"])
street = to_str(df["addr:street"])
hn = to_str(df["addr:housenumber"])

fallback_name = street.where(street.isna(), street + " " + hn.fillna(""))

df_clean["clinic_name"] = (
    name
    .fillna(operator)
    .fillna(fallback_name)
)

## 5. Address (single field)

In [5]:
postcode = clean_postcode(df["addr:postcode"])
city = "Berlin"

df_clean["address"] = (
    street
    + " "
    + hn.fillna("")
    + ", "
    + postcode.fillna("")
    + " "
    + city
).str.strip()

## 6. Administrative fields

In [6]:
df_clean["district"] = to_str(df["district"])
df_clean["district_id"] = to_str(df["district_id"])

df_clean["neighborhood"] = to_str(df["neighborhood"])
df_clean["neighborhood_id"] = pad_four_digits(df["neighborhood_id"])

df_clean["lor_id"] = to_str(df["lor_id"])

## 7. Geometry & coordinates

In [7]:
df_clean["latitude"] = df["lat"]
df_clean["longitude"] = df["lon"]
df_clean["geometry"] = df["geometry"]

## 8. Operating hours

In [8]:
df_clean["operating_hours"] = to_str(df["opening_hours"])

## 9. Contact info (consolidated)

In [9]:
contact_parts = [
    to_str(df.get("phone")),
    to_str(df.get("email")),
    to_str(df.get("website")),
]

df_clean["contact_info"] = (
    pd.concat(contact_parts, axis=1)
    .apply(lambda x: "; ".join(x.dropna()), axis=1)
    .replace("", pd.NA)
)

## 10. Accessibility

In [10]:
wheelchair = to_str(df.get("wheelchair"))
wheel_desc = to_str(df.get("wheelchair:description"))

df_clean["accessibility_features"] = (
    wheelchair
    .fillna("")
    + " "
    + wheel_desc.fillna("")
).str.strip().replace("", pd.NA)

## 11. Metadata & quality

In [11]:
df_clean["data_source"] = "OSM amenity=veterinary, Berlin"
df_clean["source_osm_id"] = to_str(df["source_osm_id"])

df_clean["has_minimum_info"] = (
    df_clean["clinic_name"].notna()
    & df_clean["latitude"].notna()
    & df_clean["longitude"].notna()
)

In [12]:
# Reverse Geocoding of Missing Addresses (Nominatim)
# -----------------------------------------------
# Targets only rows with missing addresses and valid coordinates
# Uses latitude and longitude for lookup
# Respects Nominatim API usage policies
# Respect Nominatim API rate limits (CRITICAL)
# Leaves address as NULL if no result is found

In [13]:
from geopy.geocoders import Nominatim
from time import sleep

geolocator = Nominatim(user_agent="vet_clinics_pipeline", timeout=10)

mask_missing_address = (
    df_clean["address"].isna()
    & df_clean["latitude"].notna()
    & df_clean["longitude"].notna()
)

df_missing = df_clean[mask_missing_address].copy()

print("Rows with missing address and valid coords:", len(df_missing))
print(f"Starting reverse geocoding for {len(df_missing)} rows...")

Rows with missing address and valid coords: 48
Starting reverse geocoding for 48 rows...


In [14]:
def reverse_geocode(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True)
        if location:
            return location.address
    except Exception as e:
        print(f"Error at {lat},{lon}: {e}")
    return None

In [15]:
for idx, row in df_missing.iterrows():
    address = reverse_geocode(row["latitude"], row["longitude"])
    df_clean.at[idx, "address"] = address
    sleep(1.5) # Respect Nominatim API rate limits (CRITICAL)
    print("Reverse geocoding finished.")

Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding finished.
Reverse geocoding fi

In [16]:
print("Remaining null addresses:", df_clean["address"].isna().sum())

Remaining null addresses: 0


In [17]:
df_clean["address"].isna().sum()

np.int64(0)

## 12. FINAL EXPORT

In [18]:
EXPORT_COLUMNS = [
    "id",
    "clinic_name",
    "address",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
    "lor_id",
    "latitude",
    "longitude",
    "geometry",
    "operating_hours",
    "contact_info",
    "accessibility_features",
    "data_source",
    "source_osm_id",
    "has_minimum_info",
]

In [19]:
df_export = df_clean[EXPORT_COLUMNS].copy()

In [20]:
assert df_export["address"].isna().sum() == 0, "There are still null addresses"

In [21]:
output = Path("scripts/vet_clinics_berlin_export_latest_v1.csv")
df_export.to_csv(output, index=False)

print("Export table columns:", df_export.columns.tolist())
print("Export table shape:", df_export.shape)
print("Exported reduced export table to:")
print(" -", output)
print("Final dataset ready for ingestion.")

Export table columns: ['id', 'clinic_name', 'address', 'district', 'district_id', 'neighborhood', 'neighborhood_id', 'lor_id', 'latitude', 'longitude', 'geometry', 'operating_hours', 'contact_info', 'accessibility_features', 'data_source', 'source_osm_id', 'has_minimum_info']
Export table shape: (175, 17)
Exported reduced export table to:
 - scripts/vet_clinics_berlin_export_latest_v1.csv
Final dataset ready for ingestion.


In [22]:
# Final sanity checks before commit / PR

# neighborhood_id must be 4-digit string
assert df_export["neighborhood_id"].dtype == "string", "neighborhood_id is not string"
assert df_export["neighborhood_id"].str.len().eq(4).all(), "neighborhood_id is not 4 digits"

# id must exist and be non-null
assert df_export["id"].notna().all(), "Some IDs are NULL"

# geometry must be present
assert "geometry" in df_export.columns, "geometry column missing"

# no duplicated columns
assert not df_export.columns.duplicated().any(), "Duplicated columns detected"

In [23]:
import pandas as pd

df = pd.read_csv(
    "scripts/vet_clinics_berlin_export_latest_v1.csv",
    dtype={
        "id": "string",
        "district_id": "string",
        "neighborhood_id": "string"
    }
)

df.isna().sum()

id                         0
clinic_name                4
address                    0
district                   0
district_id                0
neighborhood               0
neighborhood_id            0
lor_id                     0
latitude                   0
longitude                  0
geometry                   0
operating_hours           50
contact_info              77
accessibility_features    93
data_source                0
source_osm_id              0
has_minimum_info           0
dtype: int64